In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Dot, Flatten, Dense, Concatenate, Dropout
from tensorflow.keras.optimizers import Adam


D:\Deep Learning 2 (Final)\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [6]:
# Load data
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

print("Movies Data:")
print(movies_df.head())
print("\nRatings Data:")
print(ratings_df.head())


Movies Data:
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  

Ratings Data:
   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      306     3.5  1147868817
2       1      307     5.0  1147868828
3       1      665     5.0  1147878820
4       1      899     3.5  1147868510


In [7]:
# Preprocessing
# Encode User IDs
user_ids = ratings_df['userId'].unique().tolist()
user2user_encoded = {x: i for i, x in enumerate(user_ids)}
userencoded2user = {i: x for i, x in enumerate(user_ids)}
ratings_df['user'] = ratings_df['userId'].map(user2user_encoded)
num_users = len(user2user_encoded)

# Encode Movie IDs
movie_ids = ratings_df['movieId'].unique().tolist()
movie2movie_encoded = {x: i for i, x in enumerate(movie_ids)}
movieencoded2movie = {i: x for i, x in enumerate(movie_ids)}
ratings_df['movie'] = ratings_df['movieId'].map(movie2movie_encoded)
num_movies = len(movie2movie_encoded)

print(f"Number of users: {num_users}")
print(f"Number of movies: {num_movies}")


Number of users: 162541
Number of movies: 59047


In [8]:
# Prepare X and y
X = ratings_df[['user', 'movie']].values
y = ratings_df['rating'].values

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")


Training data shape: (22500085, 2)
Testing data shape: (2500010, 2)


In [9]:
# Model Architecture
embedding_size = 50

# User Embedding
user_input = Input(shape=(1,), name='user_input')
user_embedding = Embedding(num_users, embedding_size, name='user_embedding')(user_input)
user_vec = Flatten(name='user_flatten')(user_embedding)

# Movie Embedding
movie_input = Input(shape=(1,), name='movie_input')
movie_embedding = Embedding(num_movies, embedding_size, name='movie_embedding')(movie_input)
movie_vec = Flatten(name='movie_flatten')(movie_embedding)

# Dot Product (Matrix Factorization approach)
prod = Dot(axes=1, name='dot_product')([user_vec, movie_vec])

# Define the model
model = Model(inputs=[user_input, movie_input], outputs=prod)
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_input         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 50)     │  8,127,050 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_embedding     │ (None, 1, 50)     │  2,952,350 │ movie_input[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_flatten        │ (None, 50)        │          0 │ user_embedding[0… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ movie_flatten       │ (None, 50)        │          0 │ movie_embedding[… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_product (Dot)   │ (None, 1)         │          0 │ user_flatten[0][… │
│                     │                   │            │ movie_flatten[0]… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 11,079,400 (42.26 MB)

 Trainable params: 11,079,400 (42.26 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the model
history = model.fit(
    x=[X_train[:, 0], X_train[:, 1]],
    y=y_train,

    epochs=10,
    verbose=1,
    validation_data=([X_test[:, 0], X_test[:, 1]], y_test)
)


Epoch 1/10
 27842/703128 ━━━━━━━━━━━━━━━━━━━━ 3:40:45 20ms/step - loss: 12.0744

In [ ]:
# Plot training history
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()


In [ ]:
# Recommendation Function
def recommend_movies(user_id, num_recommendations=10):
    # Get movies watched by the user
    movies_watched_by_user = ratings_df[ratings_df.userId == user_id]

    # Get movies not watched by the user
    movies_not_watched = movies_df[~movies_df['movieId'].isin(movies_watched_by_user.movieId.values)]['movieId']
    movies_not_watched = list(set(movies_not_watched).intersection(set(movie2movie_encoded.keys())))

    if not movies_not_watched:
        print("User has watched all movies or no movies available to recommend.")
        return

    movies_not_watched_encoded = [[movie2movie_encoded.get(x)] for x in movies_not_watched]
    user_encoder = user2user_encoded.get(user_id)

    if user_encoder is None:
        print("User ID not found in training data.")
        return

    user_movie_array = np.hstack(
        ([[user_encoder]] * len(movies_not_watched), movies_not_watched_encoded)
    )

    # Predict ratings for unwatched movies
    ratings = model.predict([user_movie_array[:, 0], user_movie_array[:, 1]]).flatten()

    # Get top rated movies
    top_ratings_indices = ratings.argsort()[-num_recommendations:][::-1]
    recommended_movie_ids = [movieencoded2movie.get(movies_not_watched_encoded[x][0]) for x in top_ratings_indices]

    print(f"Top {num_recommendations} movie recommendations for user {user_id}:")
    recommended_movies = movies_df[movies_df['movieId'].isin(recommended_movie_ids)]
    return recommended_movies


In [ ]:
# Example Usage: Recommend movies for a specific user
# Replace 1 with a valid user ID from your dataset
user_id_to_recommend = ratings_df['userId'].iloc[0]
recommendations = recommend_movies(user_id=user_id_to_recommend)
print(recommendations)
